In [20]:
library(jsonlite)

samples = ("5dbf3203-ce73-41e4-bf9a-32fc856f73f5")

pcawg_info = read.table("/srv/home/mlef0011/Phasomix/rawdata/PCAWG/info/pcawg-data-releases.tsv", header=T, sep = "\t")

json_data <- fromJSON("/srv/home/mlef0011/Phasomix/rawdata/PCAWG/info/file-manifest.json", flatten = TRUE)

bam_dir <- "/srv/home/mlef0011/Phasomix/rawdata/PCAWG/bam/"

normal_bam_files = unlist(sapply(samples, function(sample) {
    matching_row <- pcawg_info[grepl(sample, pcawg_info$tumor_wgs_aliquot_id), ]
    project_code <- matching_row$dcc_project_code    
    if (strsplit(project_code, "-")[[1]][2] == "US") {
        aliquots <- unlist(strsplit(matching_row$tumor_wgs_aliquot_id, ","))
        index <- which(aliquots == sample)
        bam <- unlist(strsplit(matching_row$normal_wgs_bwa_alignment_bam_file_name, ","))[index]
        return(bam)
    } else {
        return(NA)
    }
}, simplify = FALSE))  # simplification pour éviter de convertir NULL en liste
normal_bam_files =normal_bam_files[!is.na(normal_bam_files)]

#normal_bam_files = normal_bam_files[!file.exists(file.path(bam_dir, normal_bam_files))]


tumour_bam_files = unlist(sapply(samples, function(sample) {
    matching_row <- pcawg_info[grepl(sample, pcawg_info$tumor_wgs_aliquot_id), ]
    project_code <- matching_row$dcc_project_code    
    if (strsplit(project_code, "-")[[1]][2] == "US") {
        aliquots <- unlist(strsplit(matching_row$tumor_wgs_aliquot_id, ","))
        index <- which(aliquots == sample)
        bam <- unlist(strsplit(matching_row$tumor_wgs_bwa_alignment_bam_file_name, ","))[index]
        return(bam)
    } else {
        return(NA)
    }
}, simplify = FALSE))  # simplification pour éviter de convertir NULL en liste
tumour_bam_files =tumour_bam_files[!is.na(tumour_bam_files)]

#tumour_bam_files = tumour_bam_files[!file.exists(file.path(bam_dir, tumour_bam_files))]

bam_files = c(tumour_bam_files, normal_bam_files)
if (length(bam_files) == 0) {
    message("All BAMs already downloaded.")
    quit(save = "no")
}

PCAWG_BAM = data.frame(PCAWG_ID = names(bam_files),
           bam_ID = bam_files)
GUIDs = unlist(sapply(bam_files, function(bam_file) {
    matching_row <- json_data[grepl(bam_file, json_data$file_name), ]
    bams <- unlist(strsplit(matching_row$file_name, ","))
    index <- which(bams == bam_file)
    guid <- unlist(strsplit(matching_row$object_id, ","))[index]
    return(guid)
}))

GUIDs
PCAWG_GUID = data.frame(PCAWG_ID = names(GUIDs),
       GUID = GUIDs)  

ids_info_df = merge(PCAWG_BAM, PCAWG_GUID, by = c("PCAWG_ID"), all.x = TRUE)

filtered_json <- json_data[json_data$object_id %in% ids_info_df$GUID & json_data$file_name %in% ids_info_df$bam_ID , ]
write_json(filtered_json, "/srv/home/mlef0011/VDARK/rawdata/PCAWG_sample_file-manifest.json", pretty = TRUE, auto_unbox = TRUE)

5dbf3203-ce73-41e4-bf9a-32fc856f73f5   5dbf3203-ce73-41e4-bf9a-32fc856f73f5 
"c8272932-38e0-4d6e-a4cf-87f0565e76a8" "57ac8339-b2b0-41e6-9373-090527aa4586"

In [21]:
bam_files

5dbf3203-ce73-41e4-bf9a-32fc856f73f5 
"PCAWG.bf9e4936-6222-42eb-aecd-2f15c6811b1a.bam" 
            5dbf3203-ce73-41e4-bf9a-32fc856f73f5 
"PCAWG.8cfe9d6a-b412-47e8-8eee-e84b02f76c0a.bam"